# 4. AIND Ephys Postprocessing

Build an `AINDEPhysPostprocessingScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysPostprocessingTask` for each coordinate. The task itself clones [aind-ephys-postprocessing](https://github.com/AllenNeuralDynamics/aind-ephys-postprocessing) on first use, patches `run_capsule.py` for spikeinterface API drift (`qm_params` → `metric_params`), seeds its `data/` with the preprocessing + spike-sorting outputs, writes a `params_obi.json`, runs `code/run_capsule.py`, and copies the resulting `postprocessed_<name>.zarr` (a `SortingAnalyzer`) into the single config's `coordinate_output_root`.

We read the 8-second preprocessing run from `obi-output/02_aind_ephys_preprocessing/grid_scan/0/` and the KS4 sorting from `obi-output/03_aind_ephys_spikesort_kilosort4/grid_scan/0/`. The `quality_metrics` block lowers the bin sizes that assume minute-long recordings.

## Imports and prerequisites

In [1]:
import subprocess
import sys
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.aind_ephys._04_postprocessing.blocks import QualityMetrics

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "aind-data-schema", "scikit-learn", "numba",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Resolved 21 packages in 10ms
Uninstalled 2 packages in 7ms
Installed 2 packages in 3ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema', 'scikit-learn', 'numba'], returncode=0)

## Build the scan config

In [3]:
preprocessing_output_path = (
    Path.cwd() / "../../../obi-output/02_aind_ephys_preprocessing/grid_scan/0"
).resolve()
spikesort_output_path = (
    Path.cwd() / "../../../obi-output/03_aind_ephys_spikesort_kilosort4/grid_scan/0"
).resolve()
for p in (preprocessing_output_path, spikesort_output_path):
    assert p.exists(), f"{p} not found. Run 03_aind_ephys_spikesort_kilosort4.ipynb first."

scan_config = obi.AINDEPhysPostprocessingScanConfig(
    initialize=obi.AINDEPhysPostprocessingScanConfig.Initialize(
        preprocessing_output_path=preprocessing_output_path,
        spikesort_output_path=spikesort_output_path,
        n_jobs=1,
    ),
    quality_metrics=QualityMetrics(
        presence_ratio_bin_duration_s=1.0,
        firing_range_bin_size_s=1.0,
        amplitude_cv_min_num_bins=1,
        amplitude_cv_average_num_spikes_per_bin=5,
    ),
)

## Generate the grid scan and run the postprocessing task

In [4]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/04_aind_ephys_postprocessing/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 19:03:02,838] INFO: Seeded 1 preprocessed + 1 spikesorted recording(s) into /tmp/aind-ephys-postprocessing/data


[2026-04-29 19:03:02,839] INFO: Running python -u run_capsule.py --params params_obi.json --n-jobs 1


[2026-04-29 19:05:05,192] INFO: Running postprocessing with the following parameters:
	USE_MOTION_CORRECTED: False
	N_JOBS: -1

POSTPROCESSING
Found 0 json configurations
	Processing block0_None_recording1
	Loaded binary recording from JSON
	Loading lazy recording from JSON
	Creating sorting analyzer
NumExpr defaulting to 14 threads.
	Number of original units: 14 -- Number of units after de-duplication: 11
	Setting temporary binary recording
	Computing all postprocessing extensions
	Computing quality metrics
	Saving SortingAnalyzer to zarr
POSTPROCESSING time: 69.6s



[PosixPath('../../../obi-output/04_aind_ephys_postprocessing/grid_scan/0')]

## Load the SortingAnalyzer and inspect quality metrics

In [5]:
import spikeinterface.full as si

coord_dir = Path(grid_scan.single_configs[0].coordinate_output_root)
print("coordinate_output_root:", coord_dir)

zarr_dirs = [
    p for p in coord_dir.iterdir()
    if p.is_dir() and p.name.startswith("postprocessed_") and p.suffix == ".zarr"
]
print("Postprocessed analyzers:", [p.name for p in zarr_dirs])

if zarr_dirs:
    analyzer = si.load_sorting_analyzer(zarr_dirs[0])
    print(analyzer)
    print("Extensions computed:", sorted(analyzer.get_loaded_extension_names()))

    qm_ext = analyzer.get_extension("quality_metrics")
    if qm_ext is not None:
        qm = qm_ext.get_data()
        print()
        print(qm)

coordinate_output_root: ../../../obi-output/04_aind_ephys_postprocessing/grid_scan/0
Postprocessed analyzers: ['postprocessed_block0_None_recording1.zarr']


SortingAnalyzer: 70 channels - 11 units - 1 segments - zarr - sparse
Loaded 13 extensions: correlograms, isi_histograms, noise_levels, principal_components, quality_metrics, random_spikes, spike_amplitudes, spike_locations, template_metrics, template_similarity, templates, unit_locations, waveforms
Extensions computed: ['correlograms', 'isi_histograms', 'noise_levels', 'principal_components', 'quality_metrics', 'random_spikes', 'spike_amplitudes', 'spike_locations', 'template_metrics', 'template_similarity', 'templates', 'unit_locations', 'waveforms']

    amplitude_cutoff  amplitude_cv_median  amplitude_cv_range  \
0                NaN             0.095639            0.170333   
2                NaN                  NaN                 NaN   
3                NaN             0.777748            0.000000   
5                NaN             0.928824            0.016403   
6                NaN             0.222591            0.209156   
8                NaN             0.272823          

/Users/james/Documents/obi/code/obi-main/obi-one/.venv/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:1060: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
/Users/james/Documents/obi/code/obi-main/obi-one/.venv/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:1084: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
